# Theo dõi quá trình train C2PhyNet (chạy song song với file .bat)

**Chỉ mở notebook này trên máy 1.** Mọi cell chạy trên **CPU**, không dùng GPU đang train.

- Chạy các cell **từ trên xuống**. Sau mỗi epoch, chỉ cần chạy lại **cell 2 và cell 3** để cập nhật số liệu và biểu đồ.
- Nên chạy cell 2 **ngay sau khi** cửa sổ train in dòng `Epoch X/100 | loss=...`, vì đó là lúc an toàn nhất để sao chép checkpoint.
- Notebook này **không** train. Việc train vẫn chạy bằng `run_may1.bat` / `run_may2.bat`.

In [ ]:
# Cell 1: chuẩn bị
import os, shutil, torch
import matplotlib.pyplot as plt

os.chdir(r"E:\C2PhyNet")      # thư mục chứa train_ddp2.py, encoder.pth, decoder.pth, checkpoints
torch.set_num_threads(4)        # chừa CPU cho các worker đọc dữ liệu của chương trình train
import train_ddp2 as T          # chỉ nạp class/hàm, không chạy train (khối __main__ không chạy khi import)

In [ ]:
# Cell 2: sao chép checkpoint mới nhất ra bản riêng rồi đọc (không đụng file đang được train ghi)
SRC = os.path.join("checkpoints", "checkpoint_latest.pt")
DST = os.path.join("checkpoints", "_view.pt")
shutil.copy(SRC, DST)

ckpt = torch.load(DST, map_location="cpu")
history = ckpt["history"]
print(f"Checkpoint ở epoch {ckpt['epoch'] + 1} | đã xong trọn epoch: {ckpt['epoch_finished']} | encoder/decoder đóng băng: {ckpt['frozen']}")
print(f"Số epoch đã hoàn thành trong history: {len(history)}\n")
for h in history:
    print(f"Epoch {h['epoch'] + 1:3d} | loss={h['l_total']:.4f} mse={h['mse']:.4f} mae={h['mae']:.4f} "
          f"ssim={h['ssim']:.4f} psnr={h['psnr']:.2f} | {h['time_sec'] / 60:.1f} phút")

In [ ]:
# Cell 3: vẽ diễn biến các metric theo epoch và lưu ảnh để đưa vào báo cáo
# history của train() chỉ lưu l_total, không lưu riêng l_pred và l_moment. Vì lambda_moment = 1 và không dùng SSIM loss:
#   L_pred = MSE, và L_moment = L_total - L_pred  (tính lại để vẽ đủ ô loss)
hist_plot = [dict(h, l_pred=h["mse"], l_moment=max(h["l_total"] - h["mse"], 1e-12)) for h in history]
fig = T.plot_training_history(hist_plot)
if fig is not None:
    fig.savefig("training_history.png", dpi=150, bbox_inches="tight")
    print("Đã lưu training_history.png")

## Xem thử ảnh dự đoán (tuỳ chọn, chạy trên CPU)

Cell 4 dựng model từ checkpoint trên CPU. Cell 5 chỉ quét các cơn bão **năm 2023** cho nhanh. Cell 6 dự đoán một mẫu, mất khoảng vài giây đến vài chục giây. Muốn xem mẫu khác thì đổi `idx` rồi chạy lại cell 6.

In [ ]:
# Cell 4: dựng model từ checkpoint (trên CPU)
cfg = T.Config()
cfg.data.root_dir = r"E:\minhan_storm_trajectory_final_v2\digital_typhoon_datasets\image_png"
model = T.build_model(cfg)             # nạp encoder/decoder CAE, sau đó bị ghi đè bởi trọng số trong checkpoint
model.load_state_dict(ckpt["model"])
model.eval()
print("Đã nạp trọng số từ checkpoint epoch", ckpt["epoch"] + 1)

In [ ]:
# Cell 5: tập mẫu nhỏ (chỉ các cơn bão năm 2023)
ds = T.build_typhoon_sequences(cfg.data.root_dir, cfg.data.n_in, cfg.data.n_out, cfg.data.stride,
                               cfg.data.image_size, max_median_gap_hours=3.0, year_range=(2023, 2023))
print(len(ds), "sequence")

In [ ]:
# Cell 6: dự đoán 1 mẫu và so với ảnh thật
idx = 0                                  # đổi số này để xem mẫu khác
x, y = ds[idx]                           # x: [8,1,H,W], y: [1,H,W]
with torch.no_grad():
    pred = model(x.unsqueeze(0))[0]      # [1,H,W]
m = T._batch_metrics(pred.unsqueeze(0), y.unsqueeze(0))

panels = [(x[-1, 0], "Khung vào cuối (t)", "gray", 1.0),
          (pred[0],  "Dự đoán (t+1)",      "gray", 1.0),
          (y[0],     "Thực tế (t+1)",      "gray", 1.0),
          ((pred[0] - y[0]).abs(), "|Sai số|", "Blues", None)]
fig, axes = plt.subplots(1, 4, figsize=(17, 4.6))
for ax, (img, title, cmap, vmax) in zip(axes, panels):
    im = ax.imshow(img.numpy(), cmap=cmap, vmin=0, vmax=vmax)
    ax.set_title(title, fontsize=11, loc="left")
    ax.axis("off")
fig.colorbar(im, ax=axes[-1], fraction=0.046, pad=0.02)
fig.suptitle(f"Mẫu {idx}  |  MSE={m['mse']:.4f}   MAE={m['mae']:.4f}   SSIM={m['ssim']:.3f}   PSNR={m['psnr']:.2f} dB", fontsize=12)
plt.tight_layout()
plt.show()